# 04 - Data tests

**You will learn**: generic tests (`unique`, `not_null`, `accepted_values`, `relationships`), test severity, `store_failures`,
singular tests, and `dbt_utils` tests.

**You will build**: tests on sources and silver models, and **cleaned** order models.

In dbt a **test is a `select` that returns the rows that break a rule**. Zero rows = the test passes.

| Kind | Where you write it | Example |
|---|---|---|
| **Generic** | in YAML, under a column or model | `unique`, `not_null`, `accepted_values`, `relationships` |
| **Singular** | a `.sql` file in `tests/` | any query that returns the bad rows |
| **Package** | generic tests from packages | `dbt_utils.accepted_range` |

In [ ]:
from helpers import *

## 1. Test the raw data

Tests can be attached to **sources** too. Open `models/silver/_sources.yml` and add a test to the column `order_id` of `sales_orders`:

```yaml
      - name: sales_orders
        columns:
          - name: order_id
            data_tests:
              - unique
              - not_null
```

(`columns:` goes under the table, at the same level as `description:`.) Then run the tests of that source:

In [ ]:
dbt("test --select source:bronze.sales_orders")

The `unique` test **fails** because of the duplicate orders we spotted in notebook 02.
We do not want the whole project to stop because of known raw-data problems, so for sources we set the **severity** to `warn`:

```yaml
              - unique:
                  config:
                    severity: warn
```

`error` (default) stops the run and skips everything downstream; `warn` reports and continues.
The full set of source tests, all as warnings, is provided. Run the cell to replace your file, then run the tests.

In [ ]:
%%writefile ../../src/models/silver/_sources.yml
version: 2

# Bronze = raw tables written by the notebook src/notebooks/generate_bronze_data.py.
# Tests on sources use severity `warn`: the raw data is known to contain a few bad rows,
# silver is where they get fixed.
sources:
  - name: bronze
    description: Raw tables of the AlpSport source system, one row per record as delivered.
    database: bronze
    schema: sports_shop
    tables:
      - name: products
        description: Product catalog (current state, updated in place by the source system).
        columns:
          - name: product_id
            data_tests:
              - unique: {config: {severity: warn}}
              - not_null: {config: {severity: warn}}
      - name: stores
        description: The 11 physical stores and the online shop.
        columns:
          - name: store_id
            data_tests:
              - unique: {config: {severity: warn}}
              - not_null: {config: {severity: warn}}
      - name: sales_persons
        description: Sales persons and the store they currently work in.
        columns:
          - name: sales_person_id
            data_tests:
              - unique: {config: {severity: warn}}
              - not_null: {config: {severity: warn}}
      - name: customers
        description: Customers with their loyalty tier.
        columns:
          - name: customer_id
            data_tests:
              - unique: {config: {severity: warn}}
              - not_null: {config: {severity: warn}}
      - name: sales_orders
        description: Order headers. Contains duplicates, inconsistent status casing and orphan customer ids.
        columns:
          - name: order_id
            data_tests:
              - unique: {config: {severity: warn}}
              - not_null: {config: {severity: warn}}
          - name: status
            data_tests:
              - accepted_values:
                  arguments:
                    values: ['completed', 'cancelled', 'returned']
                  config: {severity: warn}
          - name: customer_id
            data_tests:
              - relationships:
                  arguments:
                    to: source('bronze', 'customers')
                    field: customer_id
                  config: {severity: warn}
      - name: sales_order_lines
        description: Order lines. Contains duplicates, negative quantities and orphan product ids.
        columns:
          - name: order_line_id
            data_tests:
              - unique: {config: {severity: warn}}
              - not_null: {config: {severity: warn}}
          - name: quantity
            data_tests:
              - dbt_utils.accepted_range:
                  arguments:
                    min_value: 1
                  config: {severity: warn}
          - name: product_id
            data_tests:
              - relationships:
                  arguments:
                    to: source('bronze', 'products')
                    field: product_id
                  config: {severity: warn}


In [ ]:
dbt("test --select source:bronze")

Each `WARN <n>` is the number of bad rows. That is our inventory of the raw data problems:
duplicated orders and lines, orders with an unknown customer, lines with an unknown product, non-positive quantities and wrong statuses.

## 2. Look at the failing rows

A test only gives you a count. Use `--store-failures` to save the failing rows in a table (in `silver`, in the schema `<your_schema>_dbt_test__audit`):

In [ ]:
dbt("test --select source:bronze.sales_orders,test_name:unique --store-failures")

In [ ]:
audit = f"silver.{SCHEMA}_dbt_test__audit"
display(q(f"SHOW TABLES IN {audit}"))
q(f"SELECT * FROM {audit}.source_unique_bronze_sales_orders_order_id ORDER BY n_records DESC LIMIT 10")

The stored table lists each duplicated `order_id` and how many times it appears.
You can also open the compiled query in `target/compiled/...` and run it yourself.

## 3. Fix the data in silver

Bronze stays raw. **Silver is where we clean**. Two rules:

* **Orders**: keep one row per `order_id` (the latest version), normalise `status` to lower case,
  and set `customer_id` to null (and flag it) when the customer does not exist.
* **Order lines**: keep one row per `order_line_id`, drop lines with `quantity <= 0` or with an unknown product.

To remove duplicates the classic pattern is `row_number()` over a partition:

```sql
row_number() over (partition by order_id order by updated_at desc) as row_num
...
where row_num = 1
```

**Exercise A.** Complete `stg_sales_orders`. Edit the file after running the cell.

In [ ]:
%%writefile ../../src/models/silver/stg_sales_orders.sql
-- Cleans the raw orders:
--   * removes duplicates (keeps the most recently updated version of each order)
--   * normalizes the status (COMPLETED -> completed)
--   * flags orders pointing to a customer that does not exist and clears the reference
with orders as (

    select
        *,
        -- TODO 1: number the rows of each order_id, most recent updated_at (then _ingested_at) first
        1 as row_num
    from {{ source('bronze', 'sales_orders') }}

),

deduplicated as (

    select * from orders
    -- TODO 2: keep only row_num = 1

)

select
    o.order_id,
    o.order_date,
    cast(o.order_date as date) as order_day,
    c.customer_id,
    o.customer_id is not null and c.customer_id is null as is_orphan_customer,
    o.store_id,
    o.sales_person_id,
    o.channel,
    o.payment_method,
    o.status,  -- TODO 3: lower(trim(o.status)) as status
    o.created_at,
    o.updated_at,
    o._batch_id
from deduplicated as o
left join {{ ref('stg_customers') }} as c
    on o.customer_id = c.customer_id


**Exercise B.** Same for the order lines: deduplicate, keep positive quantities and known products (inner join on `stg_products`).

In [ ]:
%%writefile ../../src/models/silver/stg_sales_order_lines.sql
with lines as (

    select
        *,
        -- TODO 1: row_number() per order_line_id (latest _ingested_at first)
        1 as row_num
    from {{ source('bronze', 'sales_order_lines') }}

)

select
    l.order_line_id,
    l.order_id,
    l.product_id,
    l.quantity,
    l.unit_price,
    l.discount_pct,
    l._batch_id
from lines as l
-- TODO 2: inner join {{ ref('stg_products') }} as p on the product id (drops unknown products)
-- TODO 3: where row_num = 1 and quantity > 0


## 4. Test the silver models

Now the tests with the default severity `error`: silver must be clean. The file `models/silver/_silver.yml` is provided,
read it: it uses all four generic tests plus `dbt_utils.accepted_range`.

In [ ]:
%%writefile ../../src/models/silver/_silver.yml
version: 2
models:
- name: stg_products
  columns:
  - name: product_id
    data_tests:
    - unique
    - not_null
  - name: category
    data_tests:
    - not_null
    - accepted_values:
        arguments:
          values:
          - Ski & Snowboard
          - Running
          - Cycling
          - Hiking & Outdoor
          - Swimming
          - Fitness
          - Team Sports
          - Apparel
  - name: list_price
    data_tests:
    - dbt_utils.accepted_range:
        arguments:
          min_value: 0
          inclusive: false
  - name: unit_cost
- name: stg_stores
  columns:
  - name: store_id
    data_tests:
    - unique
    - not_null
  - name: store_type
    data_tests:
    - accepted_values:
        arguments:
          values:
          - city
          - mountain
          - outlet
          - online
- name: stg_sales_persons
  columns:
  - name: sales_person_id
    data_tests:
    - unique
    - not_null
  - name: store_id
    data_tests:
    - relationships:
        arguments:
          to: ref('stg_stores')
          field: store_id
- name: stg_customers
  columns:
  - name: customer_id
    data_tests:
    - unique
    - not_null
  - name: country_code
    data_tests:
    - relationships:
        arguments:
          to: ref('countries')
          field: country_code
  - name: loyalty_tier
    data_tests:
    - accepted_values:
        arguments:
          values:
          - basic
          - silver
          - gold
          - platinum
- name: stg_sales_orders
  columns:
  - name: order_id
    data_tests:
    - unique
    - not_null
  - name: customer_id
    data_tests:
    - relationships:
        arguments:
          to: ref('stg_customers')
          field: customer_id
  - name: is_orphan_customer
  - name: store_id
    data_tests:
    - not_null
    - relationships:
        arguments:
          to: ref('stg_stores')
          field: store_id
  - name: sales_person_id
  - name: channel
    data_tests:
    - accepted_values:
        arguments:
          values:
          - online
          - in_store
  - name: status
    data_tests:
    - accepted_values:
        arguments:
          values:
          - completed
          - cancelled
          - returned
  - name: updated_at
- name: stg_sales_order_lines
  columns:
  - name: order_line_id
    data_tests:
    - unique
    - not_null
  - name: order_id
    data_tests:
    - relationships:
        arguments:
          to: ref('stg_sales_orders')
          field: order_id
  - name: product_id
    data_tests:
    - relationships:
        arguments:
          to: ref('stg_products')
          field: product_id
  - name: quantity
    data_tests:
    - dbt_utils.accepted_range:
        arguments:
          min_value: 1
  - name: unit_price
  - name: discount_pct
    data_tests:
    - dbt_utils.accepted_range:
        arguments:
          min_value: 0
          max_value: 30


`dbt build` runs models **and** their tests, in dependency order. If a model's tests fail, its downstream models are skipped:
that is how bad data is kept out of gold. Build silver:

In [ ]:
dbt("build --select tag:silver")

In [ ]:
r = q(f'''
    SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS distinct_orders,
           COUNT_IF(is_orphan_customer) AS orphan_customer_orders
    FROM silver.{SCHEMA}.stg_sales_orders''')
check("no more duplicate orders", r.rows[0] == r.distinct_orders[0], "check the row_number logic in stg_sales_orders")
r

> If a test fails, read the message, run the compiled query from `target/compiled/...` and fix the model. Failing tests are the normal way to learn here.

## 5. Tests on the seed and a singular test

The seed gets tests too (provided):

In [ ]:
%%writefile ../../src/seeds/_seeds.yml
version: 2
seeds:
- name: countries
  config:
    column_types:
      country_code: string
      is_eu: boolean
  columns:
  - name: country_code
    data_tests:
    - unique
    - not_null
  - name: country_name
    data_tests:
    - not_null
  - name: continent
  - name: region
  - name: currency_code
  - name: is_eu


**Exercise C. A singular test.** Some rules do not fit a generic test. Write the query that returns the **orders placed before the
store opened** (`order_day < opened_date`). It belongs in `tests/` and passes when it returns nothing.

In [ ]:
%%writefile ../../src/tests/assert_order_after_store_opening.sql
-- Singular test: an order can not be placed before the store opened.
-- TODO: return order_id, order_day and opened_date of the orders whose order_day is before the store's opened_date
--       (join stg_sales_orders to stg_stores on store_id)
select
    o.order_id
from {{ ref('stg_sales_orders') }} as o
where 1 = 0


In [ ]:
dbt("build --select resource_type:seed tag:silver assert_order_after_store_opening")

Try to make it fail on purpose: change the condition to `o.order_day > s.opened_date` and see the number of failing rows, then put it back.

## Recap

* A test is a query returning the bad rows. Generic tests in YAML, singular tests in `tests/`.
* `severity: warn` for known problems you monitor, `error` to block bad data.
* `dbt build` = run + test in DAG order; a failing test skips the downstream models.
* Bronze stays raw, **silver cleans**.

---

In [ ]:
# restore_checkpoint(4)